# Lab 5 — Hypergraph Neural ODEs: learning the flow of cell identity

*Fifth session of the [`notebooks/` course](README.md) on computational synthetic morphology
(after [Lab 2 — Regulomes & hypergraphs](02_regulomes_and_hypergraphs.ipynb),
[Lab 3 — Benchmarking fidelity](03_benchmarking_fidelity.ipynb),
[Lab 4 — Modularity & the Module Identifiability Index](04_modularity_identifiability.ipynb)).*

Labs 2–4 treated the regulome as a *static* object — a hypergraph, its spectrum, its modules. But a
cell type is not a wiring diagram; it is a **stable state of the dynamics that the wiring runs**.
That is the Kauffman / Huang picture: a gene regulatory network is a dynamical system
$\dot x = f(x)$, its **attractors are cell types**, and Waddington's epigenetic landscape — the
trenches and ridges a differentiating cell rolls through — is (a Lyapunov-ish surface of) that
vector field. Development is a trajectory descending the landscape; a perturbation is a kick that
can knock the trajectory into a different basin.

This lab **learns** that vector field from data. Instead of reading KO signs off a static regulome
(Lab 3) or computing topological modules (Lab 4), we fit a **latent Hypergraph Neural ODE** —
$\dot z = f_\theta(z)$, with $f_\theta$ a small neural net whose structure is *biased by the regulon
hypergraph* (parameter-sharing within regulons; in `hgx` this is `LatentHypergraphODE`) — on a
pseudotime timecourse. Two payoffs:

1. **It splits genes into a homeostatic core and a transient shell.** A fitted flow is a
   low-dimensional smooth manifold; genes whose trajectory lives *on* it (the fate-determining TFs —
   the developmental program's slow variables) are reconstructed well, genes that are *fast,
   stimulus-driven spikes* — immediate-early genes (Fos, Jun, Egr1, Arc, Npas4) — sit *off* it and
   reconstruct badly. "Rollout MSE under the fitted ODE" is thus a *functional* classifier:
   **stable structural drivers** vs **transient stress responders**. (In the kidney-injury benchmark
   the split is stark — Lhx1/Cdh1/Pax8 ≈ 0.05–0.11 vs Fos ≈ 4.4, Jun ≈ 1.5, Cd44 ≈ 0.93, Atf3 ≈ 0.80.)
2. **It is the object Lab 6 / Lab 8 control.** Optimal control is posed on the *learned* flow — given
   a target fate basin, compute an actuation schedule that steers $z$ across the separatrix. (Why on
   the *learned* ODE and not the regulome directly: Lab 3's lesson — direction transfers, magnitude
   doesn't — so you want feedback on the flow, not open-loop on static signs.)

**Needs:** `jax`, `numpy`, `matplotlib`, `optax` ( + a hand-rolled RK4 integrator — no `diffrax`
needed here; `scripts/04_temporal_dynamics.py` does the `diffrax` + `hgx.LatentHypergraphODE` +
`equinox` version). Reads `data/processed/{temporal_expression,pseudotime_centers,fate_probabilities,
perturbation_fates}.npy` + the committed `figures/{learning_regulome,regenerative_flow}_results.json`;
falls back to a synthetic toggle-switch (a genuine two-attractor system) so every cell runs.

In [ ]:
import json, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
try:
    import optax
    HAVE_OPTAX = True
except Exception:
    HAVE_OPTAX = False

jax.config.update("jax_enable_x64", False)
rng = np.random.default_rng(0)

def _find(*parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*parts)
        if p.exists():
            return p
    return None

def _loadjson(*parts):
    p = _find(*parts);  return json.loads(p.read_text()) if p else None

# ---- the organoid pseudotime timecourse (Pando regulome substrate) ----------------------------
PROC = _find("data", "processed")
HAVE_REAL = PROC is not None and (PROC / "temporal_expression.npy").exists()
if HAVE_REAL:
    Texpr      = np.load(PROC / "temporal_expression.npy").astype(np.float32)   # (bins, genes)  z-scored
    ts_obs     = np.load(PROC / "pseudotime_centers.npy").astype(np.float32)    # (bins,)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    tf_names   = json.loads((PROC / "tf_names.json").read_text())
    key_tf_idx = json.loads((PROC / "key_tf_indices.json").read_text())          # {TF: regulon idx}  (names only used)
    fate_probs = (np.load(PROC / "fate_probabilities.npy").astype(np.float32)
                  if (PROC / "fate_probabilities.npy").exists() else None)        # (bins, 3)  DF/VF/MH
    pert_fates = (np.load(PROC / "perturbation_fates.npy").astype(np.float32)
                  if (PROC / "perturbation_fates.npy").exists() else None)        # (8, 3)
    fsum       = json.loads((PROC / "summary.json").read_text())
    fate_names = fsum.get("fates", ["DF", "VF", "MH"]); key_tfs = fsum.get("key_tfs", list(key_tf_idx))
    g_index    = {g: i for i, g in enumerate(gene_names)}
    SRC = "Fleck et al. 2023 cerebral-organoid regulome — pseudotime-binned timecourse"
else:
    print("[note] data/processed/ not found — synthetic toggle-switch timecourse (a genuine 2-attractor system).")
    Texpr = ts_obs = gene_names = tf_names = key_tf_idx = fate_probs = pert_fates = None
    fate_names = ["A", "B"]; key_tfs = ["geneA", "geneB"]; SRC = "synthetic toggle switch"

# ---- committed dynamics benchmarks -----------------------------------------------------------
learn_regulome = _loadjson("figures", "learning_regulome_results.json")    # IEG timecourse 0h/1h/4h
regen_flow     = _loadjson("figures", "regenerative_flow_results.json")    # kidney-IRI: per-TF rollout MSE

print(f"timecourse           : {SRC}")
if HAVE_REAL:
    print(f"  bins × genes       : {Texpr.shape}   pseudotime: {np.round(ts_obs, 3)}")
    print(f"  key TFs            : {', '.join(key_tfs)}")
    print(f"  fate labels        : {fate_names}   (fate_probabilities {None if fate_probs is None else fate_probs.shape}, "
          f"perturbation_fates {None if pert_fates is None else pert_fates.shape})")
print(f"committed benchmarks : learning_regulome={bool(learn_regulome)}  regenerative_flow={bool(regen_flow)}  | optax={HAVE_OPTAX}")

## 1. Cell identity as an attractor — and why a *learned* ODE

Kauffman (1969) modelled the genome as a Boolean network and noted its dynamics settle into a small
number of cyclic/fixed states — he proposed those are **cell types**. Huang et al. (2005, 2009) made
it quantitative in real data: differentiating cells move down a high-dimensional landscape and
funnel into discrete **attractors**, exactly Waddington's "creodes" with the math filled in. So:

$$\dot x = f(x), \qquad \text{cell types} = \text{attractors of } f, \qquad \text{pseudotime} \approx \text{a trajectory of } f \text{ from a progenitor basin.}$$

We do **not** know $f$. The regulome (Labs 2–4) gives the *wiring* — who can regulate whom — but not
the rate law (the Hill exponents, the dilution rates, the cooperativities). Two responses:

- **Mechanistic** (Lab 1; the `jaxctrl` IRMA example): write down Hill ODEs for a *small*,
  hand-specified circuit — those parameters are *meant* to be interpreted, so a structural-ID check
  (Lab 7) should precede fitting them.
- **Black-box** (this lab): fit a flexible $f_\theta$ (a neural net) on the *trajectory data*, biased
  toward the regulon structure. You don't interpret $\theta$ — you interpret the **trajectories,
  the attractors, the per-gene rollout error**. This is the **Hypergraph Neural ODE**: $\dot z =
  f_\theta(z)$ in a low-dim latent $z$, $f_\theta$ a small MLP with regulon-aware weight-sharing
  (`hgx.LatentHypergraphODE`). Here we fit a transparent bare-metal version — a tiny MLP vector
  field, an RK4 integrator, and gradient descent *through* the integrator.

## 2. Fit a Neural ODE on the timecourse

We fit $\dot x = f_\theta(x)$ jointly on a curated set of genes' pseudotime trajectories (the 8 key
TFs plus a few high-variance genes), with $f_\theta$ a small MLP — small enough that it *cannot*
interpolate every wiggle, so the per-gene residual is informative. We integrate from the observed
start with RK4, save at the (irregular) pseudotime points, minimise the trajectory MSE with Adam,
and differentiate straight through the rollout (`jax.grad`). (The full version — `diffrax` adjoints,
the `hgx` regulon-structured field, `equinox`/`optax` — is `scripts/04_temporal_dynamics.py`.)

In [ ]:
# ---------- pick the genes to model ----------
if HAVE_REAL:
    var = Texpr.var(0)
    extra = [i for i in np.argsort(-var) if gene_names[i] not in key_tfs][:8]
    sel_idx   = [g_index[t] for t in key_tfs if t in g_index] + list(extra)
    sel_names = [gene_names[i] for i in sel_idx]
    Y_obs = jnp.asarray(Texpr[:, sel_idx])                # (n_bins, n_genes)
    ts    = jnp.asarray(ts_obs)
else:
    # synthetic toggle switch: ẋ = β/(1+y^n) - x , ẏ = β/(1+x^n) - y  → bistable
    def _toggle(z, beta=3.0, n=3.0):
        x, y = z
        return jnp.array([beta / (1 + y ** n) - x, beta / (1 + x ** n) - y])
    ts = jnp.linspace(0.0, 6.0, 10)
    def _roll(z0, dt=0.01):
        z = z0; out = [z0]
        for k in range(1, len(ts)):
            steps = int((ts[k] - ts[k-1]) / dt)
            for _ in range(steps):
                z = z + dt * _toggle(z)
            out.append(z)
        return jnp.stack(out)
    Y_obs = _roll(jnp.array([0.6, 2.6]))
    sel_names = ["geneA", "geneB"]; key_tfs = sel_names
n_bins, n_g = Y_obs.shape
print(f"modelling {n_g} gene trajectories over {n_bins} pseudotime points: {sel_names}")

# ---------- a tiny MLP vector field ----------
def init_params(d, h, key):
    k1, k2 = jax.random.split(key)
    s = 0.3
    return {"W1": s * jax.random.normal(k1, (h, d)) / np.sqrt(d), "b1": jnp.zeros(h),
            "W2": s * jax.random.normal(k2, (d, h)) / np.sqrt(h), "b2": jnp.zeros(d),
            "leak": jnp.float32(0.1)}

def field(p, x):
    return p["W2"] @ jnp.tanh(p["W1"] @ x + p["b1"]) + p["b2"] - p["leak"] * x   # leak → stable

def rk4(p, x, dt):
    k1 = field(p, x); k2 = field(p, x + 0.5*dt*k1); k3 = field(p, x + 0.5*dt*k2); k4 = field(p, x + dt*k3)
    return x + dt/6.0 * (k1 + 2*k2 + 2*k3 + k4)

N_SUB = 8
def rollout(p, x0):
    def seg(carry, pair):
        x, _ = carry; t0, t1 = pair
        dt = (t1 - t0) / N_SUB
        x = jax.lax.fori_loop(0, N_SUB, lambda i, xx: rk4(p, xx, dt), x)
        return (x, t1), x
    (_, _), xs = jax.lax.scan(seg, (x0, ts[0]), (ts[:-1], ts[1:]))
    return jnp.concatenate([x0[None], xs], axis=0)        # (n_bins, n_g)

def loss_fn(p):
    pred = rollout(p, Y_obs[0])
    return jnp.mean((pred - Y_obs) ** 2)

# ---------- train ----------
params = init_params(n_g, max(16, 2 * n_g), jax.random.PRNGKey(0))
t0 = time.time()
if HAVE_OPTAX:
    opt = optax.adam(3e-3); state = opt.init(params)
    @jax.jit
    def step(p, st):
        l, g = jax.value_and_grad(loss_fn)(p); upd, st = opt.update(g, st); return optax.apply_updates(p, upd), st, l
    losses = []
    for it in range(700):
        params, state, l = step(params, state); losses.append(float(l))
else:                                                     # plain SGD fallback
    grad = jax.jit(jax.grad(loss_fn)); losses = []
    for it in range(700):
        g = grad(params); params = jax.tree_util.tree_map(lambda a, b: a - 5e-3 * b, params, g)
        losses.append(float(loss_fn(params)))
pred = np.asarray(rollout(params, Y_obs[0]))
print(f"fit: {len(losses)} Adam steps in {time.time()-t0:.2f}s   loss {losses[0]:.3f} → {losses[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.0))
ax = axes[0]
show = list(range(min(6, n_g)))
cols = plt.cm.tab10(np.linspace(0, 1, len(show)))
for j, c in zip(show, cols):
    ax.plot(ts, Y_obs[:, j], "o", color=c, ms=5)
    ax.plot(ts, pred[:, j], "-", color=c, lw=1.8, label=sel_names[j])
ax.set_xlabel("pseudotime"); ax.set_ylabel("expression (z-scored)" if HAVE_REAL else "level")
ax.set_title("Neural ODE: observed (•) vs learned-flow rollout (—)"); ax.legend(fontsize=8, ncol=2)
axes[1].semilogy(losses, color="#4C72B0"); axes[1].set_xlabel("Adam step"); axes[1].set_ylabel("trajectory MSE")
axes[1].set_title(f"training loss  (final {losses[-1]:.4f})")
fig.tight_layout(); plt.show()

## 3. Stable structural drivers vs transient stress responders

The fitted flow is a *single smooth manifold*. A gene whose pseudotime trajectory the flow
reconstructs well is **on the manifold** — it's part of the slow, identity-defining program (a
fate-determining TF). A gene the flow *can't* fit — typically a fast, externally-driven spike — is
**off the manifold**: a **transient stress responder** (the immediate-early genes Fos, Jun, Egr1,
Arc, Npas4 are the canonical examples; they peak ~1 h after a stimulus and decay, with no slow-state
"cause"). So **per-gene rollout MSE under the fitted ODE is a functional classifier** of genes into a
homeostatic core and a transient shell.

This is the *dynamical* sibling of Lab 4's MII and Lab 3's fidelity:
- when the modular structure dissolves (Lab 4 — MII falls, the active-gene subhypergraph fragments at
  late pseudotime), the set of well-fit homeostatic drivers shrinks;
- the well-fit drivers are also the ones whose KO *directions* transfer across systems (Lab 3) —
  they're the part of the regulome that's stable enough to be both *predictable* and *steerable*.

Two committed illustrations, where the split is cleaner than in our toy fit:
- **`figures/regenerative_flow_results.json`** — a Hypergraph Neural ODE fit (in 1.2 s) on a mouse
  kidney ischaemia–reperfusion timecourse (Balzer 2022, 4 timepoints): the homeostatic regulators
  (Lhx1, Cdh1, Pax8, Pax2, Six1/2, Wt1, Foxc2 — MSE 0.05–0.11) cleanly separate from the injury
  markers (Fos 4.4, Jun 1.5, Cd44 0.93, Atf3 0.80).
- **`figures/learning_regulome_results.json`** — an IEG timecourse (Hrvatin 2018 V1, 0 h/1 h/4 h):
  the immediate-early genes' mean z-score goes −0.13 → **+0.06** → −0.07 (the canonical peak-at-1 h),
  late-response genes (Bdnf, Homer1, Dusp1) lag — a transient *is* a thing the slow flow shouldn't fit.

In [ ]:
# ---------- live: per-gene rollout MSE from our fitted ODE ----------
per_gene_mse = np.mean((pred - np.asarray(Y_obs)) ** 2, axis=0)
order = np.argsort(per_gene_mse)
print("per-gene rollout MSE under the fitted ODE (sorted — low = stable / on-manifold):")
for j in order:
    tag = "  ← key TF" if sel_names[j] in key_tfs else ""
    print(f"  {sel_names[j]:>12}  {per_gene_mse[j]:8.4f}{tag}")

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.9))
# (a) our fit
a = axes[0]; a.barh(range(n_g), per_gene_mse[order][::-1],
                    color=["#C44E52" if sel_names[j] not in key_tfs else "#4C72B0" for j in order[::-1]])
a.set_yticks(range(n_g)); a.set_yticklabels([sel_names[j] for j in order[::-1]], fontsize=8)
a.set_xlabel("rollout MSE"); a.set_title("This lab's fit (organoid)\nblue = key TF")
# (b) committed kidney-IRI split
if regen_flow and regen_flow.get("ode", {}).get("per_tf_mse"):
    mse = regen_flow["ode"]["per_tf_mse"]; items = sorted(mse.items(), key=lambda kv: kv[1])
    names = [k for k, _ in items]; vals = [v for _, v in items]
    transient = {"Fos", "Jun", "Junb", "Atf3", "Cd44", "Vcam1", "Havcr1", "Myc"}
    a = axes[1]; a.barh(range(len(names)), vals[::-1],
                        color=["#C44E52" if n in transient else "#55A868" for n in names[::-1]])
    a.set_yticks(range(len(names))); a.set_yticklabels(names[::-1], fontsize=8); a.set_xscale("log")
    a.set_xlabel("per-TF rollout MSE (log)"); a.set_title("Kidney IRI (Balzer 2022)\ngreen = homeostatic, red = injury")
else:
    axes[1].text(0.5, 0.5, "regenerative_flow_results.json absent", ha="center"); axes[1].axis("off")
# (c) committed IEG timecourse
if learn_regulome and learn_regulome.get("timecourse"):
    tc = learn_regulome["timecourse"]; hrs = ["0h", "1h", "4h"]
    ieg = [tc[h]["ieg_mean"] for h in hrs]; late = [tc[h]["late_mean"] for h in hrs]
    a = axes[2]; a.plot(hrs, ieg, "o-", color="#C44E52", label="immediate-early genes")
    a.plot(hrs, late, "s--", color="#dd8452", label="late-response genes"); a.axhline(0, lw=.6, color="k")
    a.set_ylabel("mean z-score"); a.set_title("Activity-induced transients\n(Hrvatin 2018 V1)"); a.legend(fontsize=8)
else:
    axes[2].text(0.5, 0.5, "learning_regulome_results.json absent", ha="center"); axes[2].axis("off")
fig.suptitle("Stable structural drivers (low rollout MSE) vs transient stress responders (high)", y=1.04)
fig.tight_layout(); plt.show()
print("\n  (Our toy organoid fit has a *modest* spread: the curated 2792-gene set is TF-and-target-curated")
print("   and excludes the classic IEGs, so the 'extra' high-variance genes stand in for the transient class —")
print("   note the 8 key TFs all cluster at the low-MSE end. The kidney panel shows the real homeostatic-vs-")
print("   injury split on per-cell data; the lesson is the *classifier*, not our numbers.)")

## 4. The Waddington picture — fates as basins, perturbations as kicks

A fitted flow *is* a landscape. To see basins and a separatrix you need a system with more than one
attractor, so we use the cleanest example — the **toggle switch** (Lab 1): $\dot x = \beta/(1+y^n)
- x$, $\dot y = \beta/(1+x^n) - y$, which is **bistable** ("$x$ high, $y$ low" vs the mirror). We
integrate it from a fan of initial conditions, colour each trajectory by which attractor it reaches,
and overlay the separatrix (the diagonal $x=y$). A "knockout" = start with that gene's level pushed
down → if the push crosses the separatrix, the cell ends in the *other* basin. That is, in miniature,
exactly what the anatomical compiler (Lab 8) does on the *learned* organoid ODE: find the actuation
that moves the latent state across the separatrix into the target fate basin.

For the real organoid, we don't re-fit a multi-attractor model here (that needs the per-cell data —
`scripts/04_temporal_dynamics.py`, 34k cells), but the data already shows the population *descending
into basins*: `fate_probabilities.npy` is the DF/VF/MH composition along pseudotime (one population
splitting three ways), and `perturbation_fates.npy` is the *shift* in that fate composition induced by
each of the 8 key-TF knockouts — the data's version of "knocking out a TF tilts the landscape."

In [ ]:
# ---------- the toggle-switch landscape: basins + separatrix + a "knockout" ----------
def toggle(z, beta=3.0, n=3.0):
    x, y = z[..., 0], z[..., 1]
    return jnp.stack([beta / (1 + y ** n) - x, beta / (1 + x ** n) - y], axis=-1)
def toggle_rollout(z0, T=8.0, dt=0.01):
    z = z0
    for _ in range(int(T / dt)):
        z = z + dt * toggle(z)
    return z
grid = np.linspace(0.05, 3.0, 11)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
# vector field
xx, yy = np.meshgrid(np.linspace(0, 3, 18), np.linspace(0, 3, 18))
uv = np.asarray(toggle(jnp.stack([jnp.asarray(xx), jnp.asarray(yy)], -1)))
ax.quiver(xx, yy, uv[..., 0], uv[..., 1], color="#bbb", angles="xy")
for x0 in grid:
    for y0 in grid:
        zf = np.asarray(toggle_rollout(jnp.array([x0, y0])))
        basin = "#4C72B0" if zf[0] > zf[1] else "#C44E52"
        ax.plot([x0], [y0], ".", color=basin, ms=6)
ax.plot([0, 3], [0, 3], "k--", lw=1, label="separatrix x = y")
# a "geneA knockout": start from a geneA-high state but push geneA down across the line
z_id = np.array([2.6, 0.5]); z_ko = np.array([0.3, 0.5])
for z0, lbl, mk in [(z_id, "wild type → A-high basin", "*"), (z_ko, "geneA KO (pushed down) → B-high basin", "P")]:
    tr = [z0]; z = jnp.asarray(z0)
    for _ in range(800): z = z + 0.01 * toggle(z); tr.append(np.asarray(z))
    tr = np.array(tr); ax.plot(tr[:, 0], tr[:, 1], lw=2.2, label=lbl); ax.plot(tr[0, 0], tr[0, 1], mk, ms=13, color=ax.lines[-1].get_color())
ax.set_xlabel("geneA"); ax.set_ylabel("geneB"); ax.set_title("Toggle-switch landscape — two basins, one separatrix\n(a KO that crosses the line switches fate)")
ax.legend(fontsize=8, loc="upper right"); ax.set_xlim(0, 3); ax.set_ylim(0, 3)

# ---------- the real organoid: the population descending into basins ----------
ax = axes[1]
if HAVE_REAL and fate_probs is not None:
    for k, nm in enumerate(fate_names):
        ax.plot(ts_obs, fate_probs[:, k], "o-", label=f"fate {nm}")
    ax.set_xlabel("pseudotime"); ax.set_ylabel("fate probability mass"); ax.set_title("Organoid: one population → three fate basins\n(fate_probabilities.npy)")
    ax.legend(fontsize=8)
elif HAVE_REAL:
    ax.text(0.5, 0.5, "fate_probabilities.npy absent", ha="center"); ax.axis("off")
else:
    ax.axis("off")
fig.tight_layout(); plt.show()

# ---------- knocking out a TF tilts the landscape ----------
if HAVE_REAL and pert_fates is not None:
    print("Δ fate-probability after each key-TF knockout (perturbation_fates.npy — signed shifts, each row ≈ sums to 0):")
    print("  " + "".join(f"{nm:>10}" for nm in (["KO"] + list(fate_names) + ["‖Δfate‖"])))
    mags = np.linalg.norm(pert_fates, axis=1)
    for i, tf in enumerate(key_tfs[: pert_fates.shape[0]]):
        print("  " + f"{tf:>10}" + "".join(f"{v:>10.3f}" for v in pert_fates[i]) + f"{mags[i]:>10.3f}")
    order_ko = np.argsort(-mags)
    j = int(order_ko[0])
    print(f"\n  biggest 'landscape tilt': KO {key_tfs[j]}  (‖Δfate‖ = {mags[j]:.3f}); next: "
          + ", ".join(f"{key_tfs[k]} ({mags[k]:.2f})" for k in order_ko[1:3]))
    print("  → these are the single-TF kicks that most shift the fate basins. Lab 8 turns it around:")
    print("    given a *target* fate composition, what actuation (which TFs, how much, when) gets you there?")

## 5. What a Hypergraph Neural ODE is — and isn't

- **It's the flow, not the parameters.** $\theta$ is deliberately *structurally non-identifiable*
  (Lab 7) — many weight settings give the same trajectories — and that's *fine*, because we never
  read $\theta$; we read the **trajectories, the attractors, the per-gene rollout MSE, the
  separatrices**. (Contrast the *mechanistic* reductions of Lab 1 / the `jaxctrl` IRMA example,
  whose Hill coefficients *are* meant to be interpreted — those need a structural-ID check *before*
  fitting.)
- **It's dynamics on top of the static regulome, not a replacement.** Labs 2–4 give the hypergraph,
  its spectrum, its modules; this lab adds *time*. The "hypergraph" in *Hypergraph* Neural ODE is a
  **prior** on $f_\theta$ (weight-sharing within regulons — `hgx.LatentHypergraphODE`), a soft bias,
  not a hard constraint; you can fit a generic latent ODE (as we did) and lose only inductive bias.
- **It's a model of the *measured* trajectory, with the usual caveats.** A 10-point binned pseudotime
  is a smoothed pseudo-trajectory of an asynchronous population; fit on per-cell data with an SDE
  (exercise (a)) you get the *branching* structure and a calibrated notion of fate uncertainty.
- **Rollout MSE is a *relative* readout.** "Stable vs transient" is a ranking within one fit on one
  dataset — don't compare the absolute MSE across systems (different scales, different #timepoints).
  The classifier transfers; the numbers don't (same lesson as Lab 3's fidelity triple, Lab 4's MII).

## 6. Exercises

**(a) ODE → SDE.** Add a learned diffusion term, $dz = f_\theta(z)\,dt + g_\phi(z)\,dW$, and fit on
the *per-cell* data (`data/processed/cell_fate_probs.npy` + the expression matrix; or the binned data
with bootstrap noise). Does the SDE *rollout variance* along pseudotime track the CellRank fate
*entropy* (high near the branch point, low in committed lineages)? (`scripts/04_temporal_dynamics.py`
does this with `diffrax`'s SDE solvers.)

**(b) Poincaré latent space.** Cell-fate decisions are *hierarchical* (a binary tree of branchings).
Embed the latent trajectory in a hyperbolic (Poincaré) disk rather than Euclidean $\mathbb R^L$ and
compare the Gromov $\delta$-hyperbolicity (Euclidean vs Poincaré). Is the fate hierarchy more
tree-like in hyperbolic space? (`scripts/04_temporal_dynamics.py`, Analysis 2.3.)

**(c) Finer time, sharper split.** Re-run §2–§3 with $T \in \{20, 50\}$ pseudotime bins (the
preprocessing has these variants). Does the stable/transient rollout-MSE split *sharpen* with more
timepoints (the flow has more to fit, so transients stick out more), or does noise swamp it?

**(d) Put the hypergraph in.** Replace the generic MLP $f_\theta$ with `hgx.LatentHypergraphODE`
(weight-sharing within regulons, seeded from the Fleck incidence). Does the per-gene rollout-MSE
ranking change — do the structured-prior "stable drivers" line up better with the regulome's master
TFs (Lab 2) and with the high-MII modules (Lab 4)?

**(e) Close the loop (→ Lab 6/6).** Hand the fitted vector field to `jaxctrl`: linearise it,
check controllability, and compute (LQR / `diffrax`-adjoint) the actuation that moves the latent
state from the progenitor basin to a chosen fate basin. This *is* the anatomical compiler in
miniature — Lab 8 does it on the real learned organoid ODE.

Starters for (a) and (d):

In [ ]:
# --- Exercise (a) starter: an SDE rollout (Euler–Maruyama) with a tiny learned-noise term -----
def init_diffusion(d, key):
    return {"log_sigma": -1.5 * jnp.ones(d)}              # state-independent noise to start
def sde_rollout(p, p_diff, x0, key, n_sub=8):
    sig = jnp.exp(p_diff["log_sigma"])
    def seg(carry, pair):
        x, k, _ = carry; t0, t1 = pair; dt = (t1 - t0) / n_sub
        def inner(i, st):
            xx, kk = st; kk, sub = jax.random.split(kk)
            xx = xx + dt * field(p, xx) + jnp.sqrt(dt) * sig * jax.random.normal(sub, xx.shape)
            return (xx, kk)
        x, k = jax.lax.fori_loop(0, n_sub, inner, (x, k))
        return (x, k, t1), x
    (_, _, _), xs = jax.lax.scan(seg, (x0, key, ts[0]), (ts[:-1], ts[1:]))
    return jnp.concatenate([x0[None], xs], 0)
p_diff = init_diffusion(n_g, jax.random.PRNGKey(1))
paths = np.stack([np.asarray(sde_rollout(params, p_diff, Y_obs[0], jax.random.PRNGKey(s))) for s in range(40)])
print(f"SDE: 40 rollouts; per-pseudotime spread (std over paths, mean over genes): "
      f"{np.round(paths.std(0).mean(1), 3)}")
print("  → your turn: fit log_sigma (or a g_φ(z) net); does the spread track CellRank fate entropy? (exercise a)")

# --- Exercise (d) starter: the hgx regulon-structured field -----------------------------------
try:
    import hgx
    print("\nhgx is importable — see hgx.LatentHypergraphODE; seed it from data/processed/incidence.npy")
    print("  (build the hypergraph with hgx.from_incidence(H), then fit dz/dt = LatentHypergraphODE(...)(z),")
    print("   compare the per-gene rollout MSE ranking to this lab's generic-MLP ranking.)")
except Exception:
    print("\n[note] hgx not importable here — see scripts/04_temporal_dynamics.py for the LatentHypergraphODE version.")

## Recap & where this sits

- A **Hypergraph Neural ODE** is the *learned vector field* of cell identity — $\dot z = f_\theta(z)$
  fit on a pseudotime timecourse, $f_\theta$ a small (regulon-biased) net. It's the flow that
  matters, not $\theta$.
- Its per-gene **rollout MSE** is a functional classifier: low ⇒ **stable structural driver** (on
  the slow, identity-defining manifold — Lhx1/Pax8/Cdh1 in the kidney-IRI fit; the fate TFs in the
  organoid); high ⇒ **transient stress responder** (off-manifold spikes — the IEGs Fos/Jun/Egr1).
  This is the *dynamical* sibling of Lab 4's MII (modules dissolve ⇒ driver set shrinks) and Lab 3's
  fidelity triple (well-fit drivers ⇒ transferable KO directions).
- The fitted flow is a **Waddington landscape**: cell types are basins, separatrices are fate
  decisions, a knockout is a kick (the toggle-switch demo makes it literal; `fate_probabilities.npy`
  and `perturbation_fates.npy` are the organoid data's version). Steering across a separatrix to a
  target basin is the **anatomical compiler** — Lab 8.
- **Next:** Lab 6 — control theory on cellular dynamics (`jaxctrl`: identify a surrogate →
  controllability → LQR → driver nodes), Lab 7 — is the *mechanistic* reduction even identifiable?,
  Lab 8 — the anatomical compiler (optimal control on this learned ODE).
- Pipeline: `scripts/04_temporal_dynamics.py` (the `diffrax` + `hgx.LatentHypergraphODE` + SDE +
  Poincaré version), `scripts/benchmark_learning_regulome.py` (the IEG/activity-induced timecourse),
  `scripts/benchmark_regenerative_flow.py` (the kidney-IRI stable-vs-transient split). The control
  worked examples live in the `jaxctrl` repo (`examples/irma_sindy_lqr.ipynb`,
  `examples/grn_hypergraph_drivers.ipynb`, `examples/repressilator_control_demo.py`).